# Course 23 lab — Human approval gates and delegated authority

This notebook moves from a weak Boolean approval to exact-subject, current-authority, single-use execution. It is credential-free and performs no external action.

## 1. The insecure baseline

A record such as `{'approved': True}` does not establish identity, role, resource scope, subject, policy, expiry, revocation, or usage. List the attacks that remain possible before continuing.

In [ ]:
from pathlib import Path
import sys
from tempfile import TemporaryDirectory

sys.path.insert(0, str(Path.cwd()))
from approval_app.control_plane import ApprovalControlPlane
from approval_app.models import ControlError
from approval_app.simulation import (NOW, LATER, FUTURE, concurrency_scenario,
                                     production_subject, run_reference)

ROOT = Path.cwd()
if not (ROOT / 'policies').exists():
    raise RuntimeError('Run this notebook from northstar-approval-control')

## 2. Bind the decision subject

The digest covers action, resource, environment, artifact, specification, policy revision, risk, and incident context. Any material mutation creates another subject.

In [ ]:
subject = production_subject()
{'change_id': subject.change_id, 'artifact': subject.artifact_digest, 'subject_digest': subject.digest}

## 3. Issue two independent receipts and execute once

The production policy requires release and AI-governance roles exercised by distinct principals. The executor receives only receipt references and re-resolves every fact.

In [ ]:
with TemporaryDirectory() as directory:
    plane = ApprovalControlPlane(Path(directory) / 'approval.db', ROOT)
    plane.create_request('REQ-NB', 'SESSION-REQUESTER', 'agent-ai-a', subject, NOW)
    release = plane.decide('REQ-NB', 'SESSION-RELEASE', 'release_authority', 'APPROVED', NOW, FUTURE)
    governance = plane.decide('REQ-NB', 'SESSION-GOVERNANCE', 'ai_governance_owner', 'APPROVED', NOW, FUTURE)
    decision = plane.execute('REQ-NB', release.receipt_ids + governance.receipt_ids, subject, 'EXEC-NB', LATER)
    exact_execution = {'state': decision.state, 'operations': plane.operation_count('REQ-NB'), 'audit_valid': plane.verify_audit_chain()}
exact_execution

## 4. Test the concurrency invariant

Two worker threads use separate database connections and race on one receipt. A correct result contains one execution, one registered operation, and one consumed-receipt denial.

In [ ]:
with TemporaryDirectory() as directory:
    race = concurrency_scenario(ROOT, directory)
race

## 5. Run the conformance and adversarial populations

Inspect every reason code. A green aggregate is useful only when the population and authority boundary are explicit. The final decision is eligibility for accountable review, not deployment.

In [ ]:
report = run_reference(ROOT)
{
    'decision': report['decision'],
    'golden': report['golden_scenarios']['passed'],
    'evaluations': report['evaluation_population']['passed'],
    'audit_tamper_detected': report['audit_integrity']['tamper_detected'],
    'non_authorities': report['non_authorities'],
}

## Reflection

1. Which fields would your organization add to the subject?
2. Which authority stores are independent of the implementation team?
3. How would you make a remote deployment idempotent and reconcilable?
4. Which emergency actions are reversible enough to pre-authorize?
5. When is a repository-native reviewer gate sufficient, and when is a custom control plane necessary?